In [1]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report



In [2]:
from google.colab import drive
drive.mount('/content/drive')
# 1. Load Daily Dataset
df = pd.read_parquet("/content/drive/MyDrive/daily_data_fixed.parquet")

# 2. Select same features as Spark version
feature_cols = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean',
    'daylight_duration', 'sunshine_duration', 'precipitation_sum', 'rain_sum', 'snowfall_sum',
    'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max',
    'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration'
]

# 3. Drop nulls
df_clean = df[['weather_code'] + feature_cols].dropna()

# 4. Encode target
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['weather_code'])

# 5. Assemble features
X = df_clean[feature_cols].values
y = df_clean['label'].values

# Scale features (optional, but keeps consistency)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# 6. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 7. Train Random Forest
rf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)

start_train = time.time()
rf.fit(X_train, y_train)
end_train = time.time()

# 8. Predict
start_pred = time.time()
y_pred = rf.predict(X_test)
end_pred = time.time()

# 9. Evaluate
accuracy = accuracy_score(y_test, y_pred)

print("=== Baseline (scikit-learn, full daily dataset) ===")
print(f"Training time: {end_train - start_train:.2f} seconds")
print(f"Prediction time: {end_pred - start_pred:.2f} seconds")
print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Mounted at /content/drive
=== Baseline (scikit-learn, full daily dataset) ===
Training time: 248.74 seconds
Prediction time: 7.19 seconds
Test Accuracy: 0.7148

Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.60      0.56     67433
           1       0.33      0.22      0.26     55206
           2       0.30      0.16      0.21     63026
           3       0.72      0.84      0.78    249127
           4       0.93      0.95      0.94    245046
           5       0.66      0.71      0.69     98500
           6       0.35      0.20      0.26     37859
           7       0.61      0.68      0.65    100544
           8       0.77      0.74      0.75     92483
           9       0.73      0.40      0.52      7828
          10       0.91      0.93      0.92     15257
          11       0.86      0.87      0.86     14156
          12       0.81      0.63      0.71      2703

    accuracy                           0.71   1049168
   

In [3]:
# --- Mount Drive (Colab) ---
from google.colab import drive
drive.mount("/content/drive")

# --- Imports ---
import os, time
import numpy as np
import pandas as pd
from datetime import datetime
import pyarrow as pa
import pyarrow.dataset as ds

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# --- Path to your hourly parquet data (single file or directory) ---
DATA_PATH = "/content/drive/MyDrive/hourly_data_fixed.parquet"  # change if needed

# --- Features (start lean; expand later if stable) ---
feature_cols = [
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'apparent_temperature','precipitation',
    'pressure_msl','cloud_cover',
    'wind_speed_10m','wind_direction_10m',
    'et0_fao_evapotranspiration'
]

# --- Reader: load ONLY one year using predicate pushdown on datetime ---
def load_hourly_slice_year(path: str, year: int, cols: list[str]) -> pd.DataFrame:
    """
    Loads rows for a given calendar year without pulling the whole dataset.
    Works for a single parquet file or a directory of parquet files.
    """
    dataset = ds.dataset(path, format="parquet")
    # read only needed columns to save RAM
    needed_cols = ['weather_code', 'datetime'] + cols

    start = datetime(year, 1, 1)
    end   = datetime(year + 1, 1, 1)
    filt = (ds.field("datetime") >= start) & (ds.field("datetime") < end)

    # Arrow Table -> Pandas (then downcast)
    table = dataset.to_table(columns=needed_cols, filter=filt)
    if table.num_rows == 0:
        return pd.DataFrame(columns=needed_cols)

    df = table.to_pandas()  # safe: only a year's slice + selected columns
    # ensure proper dtypes + downcast to save memory
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
    df['weather_code'] = pd.to_numeric(df['weather_code'], errors='coerce')
    return df

# --- Training loop: year-by-year to avoid OOM ---
# Tip: start small to verify (e.g., 1950-1952), then expand the range
YEAR_START, YEAR_END = 1950, 1955  # inclusive start, exclusive end (change later)
results = []
total_rows = 0

for yr in range(YEAR_START, YEAR_END):
    print(f"\n=== Reading year {yr} ===")
    df = load_hourly_slice_year(DATA_PATH, yr, feature_cols)
    if df.empty:
        print(f"No rows for {yr} (skipping).")
        continue

    # Clean & prepare
    df = df.dropna(subset=['weather_code'] + feature_cols)
    if df.empty:
        print(f"No rows after cleaning for {yr} (skipping).")
        continue

    # Encode target (treat as categorical)
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['weather_code'].astype(str))

    X = df[feature_cols].values
    y = df['label'].values

    # Standardize within-year (keeps memory bounded)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Model
    clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)

    # Train & time
    t0 = time.time()
    clf.fit(X_train, y_train)
    t1 = time.time()

    # Predict & time
    p0 = time.time()
    y_pred = clf.predict(X_test)
    p1 = time.time()

    acc = accuracy_score(y_test, y_pred)
    rows = len(df)
    total_rows += rows

    results.append({
        "Year": yr,
        "Rows": rows,
        "Train Time (s)": round(t1 - t0, 2),
        "Pred Time (s)": round(p1 - p0, 2),
        "Accuracy": round(acc, 4)
    })

# --- Summarize ---
results_df = pd.DataFrame(results)
print("\n=== Yearly Baseline Results ===")
print(results_df)

if not results_df.empty:
    print("\n=== Averages Across Years ===")
    print({
        "Avg Train Time (s)": round(results_df["Train Time (s)"].mean(), 2),
        "Avg Pred Time (s)": round(results_df["Pred Time (s)"].mean(), 2),
        "Avg Accuracy": round(results_df["Accuracy"].mean(), 4),
        "Total Rows Used": int(total_rows),
        "Years Processed": results_df["Year"].nunique()
    })


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== Reading year 1950 ===

=== Reading year 1951 ===

=== Reading year 1952 ===

=== Reading year 1953 ===

=== Reading year 1954 ===

=== Yearly Baseline Results ===
   Year     Rows  Train Time (s)  Pred Time (s)  Accuracy
0  1950  2049840           43.01           0.77    0.9970
1  1951  2049840           41.84           0.49    0.9973
2  1952  2055456           41.58           0.70    0.9965
3  1953  2049840           43.06           0.55    0.9974
4  1954  2049840           32.21           0.46    0.9970

=== Averages Across Years ===
{'Avg Train Time (s)': np.float64(40.34), 'Avg Pred Time (s)': np.float64(0.59), 'Avg Accuracy': np.float64(0.997), 'Total Rows Used': 10254816, 'Years Processed': 5}


In [ ]:
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pyarrow.dataset as ds

from google.colab import drive
drive.mount('/content/drive')
# Path to full hourly dataset
DATA_PATH = "/content/drive/MyDrive/hourly_data_fixed.parquet"

# Features (subset to keep memory manageable)
feature_cols = [
    'temperature_2m','relative_humidity_2m','dew_point_2m',
    'apparent_temperature','precipitation',
    'pressure_msl','cloud_cover',
    'wind_speed_10m','wind_direction_10m',
    'et0_fao_evapotranspiration'
]

print("=== Loading FULL Hourly dataset into memory ===")
dataset = ds.dataset(DATA_PATH, format="parquet")

# Only read required columns
table = dataset.to_table(columns=['weather_code'] + feature_cols)
df = table.to_pandas(split_blocks=True, self_destruct=True)  # Arrow -> Pandas

print("Data loaded. Shape:", df.shape)

# Drop nulls
df_clean = df.dropna(subset=['weather_code'] + feature_cols)

# Encode target
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['weather_code'])

# Features and labels
X = df_clean[feature_cols].astype('float32').values
y = df_clean['label'].values

# Standardize
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train Random Forest
clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)

print("\n=== Training on FULL dataset ===")
t0 = time.time()
clf.fit(X_train, y_train)
t1 = time.time()

print("\n=== Predicting ===")
p0 = time.time()
y_pred = clf.predict(X_test)
p1 = time.time()

acc = accuracy_score(y_test, y_pred)

print("\n=== Full Dataset Baseline Results ===")
print(f"Rows: {len(df_clean)}")
print(f"Training time: {t1 - t0:.2f} seconds")
print(f"Prediction time: {p1 - p0:.2f} seconds")
print(f"Test Accuracy: {acc:.4f}")


Mounted at /content/drive
=== Loading FULL Hourly dataset into memory ===
